# Geometric Transformations in Digital Image Processing

This notebook demonstrates:
1. **Rotation** (Affine Transformation)
2. **Projective Transformation**

---

## Setup - Import Libraries

In [ ]:
from PIL import Image
from numpy import array, zeros
import matplotlib.pyplot as plt

print("Libraries loaded successfully!")

## Load the Image

In [ ]:
# Load image and convert to grayscale
img1 = Image.open('parrot.jpg').convert('L')
im1 = array(img1)

rows = im1.shape[0]  # Height
cols = im1.shape[1]  # Width

print(f"Image Size: {rows} x {cols}")
plt.imshow(im1, cmap='gray')
plt.title('Original Image')
plt.show()

---

# 1. ROTATION (Affine Transformation)

## Rotation Matrix (30 degrees)

```
R = | cos(θ)  -sin(θ)   0 |
    | sin(θ)   cos(θ)   0 |
    |   0        0      1 |
```

For θ = 30°:
- cos(30°) = 0.866
- sin(30°) = 0.5

In [ ]:
# ROTATION MATRIX (30 degrees)
# cos(30) = 0.866, sin(30) = 0.5

R = [[.866, -.5, 0],
     [.5, .866, 0],
     [0, 0, 1]]

print("Rotation Matrix (30°):")
print(array(R))

In [ ]:
# Calculate output image size
tp = R @ array([[rows], [cols], [1]])
rowsd = int(tp[0,0]) + 200
colsd = int(tp[1,0]) + 200

print(f"Output Image Size: {rowsd} x {colsd}")

# Create empty output image
im2_rotation = zeros([rowsd, colsd])

In [ ]:
# FORWARD MAPPING - Apply rotation to each pixel

for r in range(rows):
    for c in range(cols):
        # Apply transformation matrix
        tp = R @ array([[r], [c], [1]])
        
        rd = int(tp[0,0])  # Destination row
        cd = int(tp[1,0])  # Destination column
        # td = tp[2,0] is always 1 for affine (no division needed)
        
        # Copy pixel if within bounds
        if rd >= 0 and cd >= 0:
            im2_rotation[rd, cd] = im1[r, c]

print("Rotation complete!")

In [ ]:
# Display results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(im1, cmap='gray')
plt.title('Original Image')

plt.subplot(1, 2, 2)
plt.imshow(im2_rotation, cmap='gray')
plt.title('Rotated 30° (Affine)')

plt.tight_layout()
plt.show()

---

# 2. PROJECTIVE TRANSFORMATION

## Projective Matrix

```
P = | a  b  c |   | 0.8    0.3    10   |
    | d  e  f | = | -0.7   0.9    30   |
    | g  h  1 |   | 0.0028 0.0036 1    |
```

**KEY DIFFERENCE**: Last row is NOT [0, 0, 1]!

This means we MUST divide by the homogeneous coordinate (w).

In [ ]:
# PROJECTIVE MATRIX
# Note: Last row has non-zero values!

P = array([[.8, .3, 10],
           [-.7, .9, 30],
           [0.0028, 0.0036, 1]])

print("Projective Matrix:")
print(P)

In [ ]:
# Calculate output image size (DIVIDE BY W!)
tp = P @ array([[rows], [cols], [1]])
rowsd = int(tp[0,0] / tp[2,0]) + 150  # Divide by w
colsd = int(tp[1,0] / tp[2,0]) + 150  # Divide by w

print(f"Output Image Size: {rowsd} x {colsd}")

# Create empty output image
im2_projective = zeros([rowsd, colsd])

In [ ]:
# FORWARD MAPPING with PERSPECTIVE DIVISION

for r in range(rows):
    for c in range(cols):
        # Apply transformation matrix
        tp = P @ array([[r], [c], [1]])
        
        rd = tp[0,0]  # x' (homogeneous)
        cd = tp[1,0]  # y' (homogeneous)
        td = tp[2,0]  # w (NOT always 1!)
        
        # CRITICAL: Convert homogeneous to Cartesian
        rd = int(rd / td)  # x = x'/w
        cd = int(cd / td)  # y = y'/w
        
        # Copy pixel if within bounds
        if rd >= 0 and cd >= 0 and rd < rowsd and cd < colsd:
            im2_projective[rd, cd] = im1[r, c]

print("Projective transformation complete!")

In [ ]:
# Display results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(im1, cmap='gray')
plt.title('Original Image')

plt.subplot(1, 2, 2)
plt.imshow(im2_projective, cmap='gray')
plt.title('Projective Transformation')

plt.tight_layout()
plt.show()

---

# Compare All Results

In [ ]:
# Side by side comparison
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(im1, cmap='gray')
plt.title('Original')

plt.subplot(1, 3, 2)
plt.imshow(im2_rotation, cmap='gray')
plt.title('Rotation (Affine)')

plt.subplot(1, 3, 3)
plt.imshow(im2_projective, cmap='gray')
plt.title('Projective')

plt.tight_layout()
plt.show()

---

# Summary - Key Differences

| Feature | Rotation (Affine) | Projective |
|---------|-------------------|------------|
| Last row | `[0, 0, 1]` | `[0.0028, 0.0036, 1]` |
| w value | Always 1 | Varies |
| Division | Not needed | **REQUIRED** |
| Parallel lines | Preserved | NOT preserved |

## Code Difference

**Rotation:**
```python
rd = int(tp[0,0])
cd = int(tp[1,0])
```

**Projective:**
```python
rd = int(tp[0,0] / tp[2,0])  # Divide by w!
cd = int(tp[1,0] / tp[2,0])  # Divide by w!
```

---

# Practice: Try Different Angles

Common rotation angles:
- 45°: cos = 0.707, sin = 0.707
- 60°: cos = 0.5, sin = 0.866
- 90°: cos = 0, sin = 1

In [ ]:
# Try 45 degree rotation
R_45 = [[0.707, -0.707, 0],
        [0.707, 0.707, 0],
        [0, 0, 1]]

im2_45 = zeros([rows+300, cols+300])

for r in range(rows):
    for c in range(cols):
        tp = R_45 @ array([[r], [c], [1]])
        rd, cd = int(tp[0,0]), int(tp[1,0])
        if rd >= 0 and cd >= 0:
            im2_45[rd, cd] = im1[r, c]

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(im1, cmap='gray')
plt.title('Original')
plt.subplot(1, 2, 2)
plt.imshow(im2_45, cmap='gray')
plt.title('Rotated 45°')
plt.show()